# D2.9 · The fleet kill switch

**Function D — AI for SecOps → The Incident Responder**  ·  *AI for Security*

Builds on **[D2.8 · Regulatory clock](https://spbreed.github.io/cyber-commons/lessons/D2.8.html)**.

| | |
|---|---|
| Open-source tooling | Vault, Kubernetes |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The third party's exposure ended when the third party revoked its keys, not when the agents were stopped. Terminating a fleet whose credentials stay valid moves the incident rather than ending it.

## 2 · The framework

```
   one selector, one action, in this order

      snapshot state + transcripts     <- or the incident is unreconstructable
              |
           terminate
              |
        revoke credentials             <- or the tokens outlive the agents

   the path must not run through the fleet
      via orchestrator API      NO   depends on the thing being stopped
      via the agent sidecar     NO   workload credentials
      out-of-band control plane YES  separate creds, separate network

   tested quarterly, under partial failure, target under 5 minutes
```

D2.4 contained one agent. This is the control for the case where the unit of
containment is the fleet.

The source incident makes the requirement concrete in one detail: third-party
access ended when the **third party** revoked its keys — not when the agents
stopped. Terminating agents while their credentials stay valid leaves the
persistence exactly where it was, and moves the incident rather than ending it.

So the kill switch is one action with four properties:

**One selector.** Experiment, model, time window, or everything. Not a runbook
of forty steps executed under pressure.

**Independent of the agent execution path.** Separate credentials, separate
network path, so a compromised fleet cannot interfere with the thing that stops
it.

**Evidence-preserving.** Snapshot state and transcripts *before* terminating.
An incident you cannot reconstruct afterwards has been survived, not handled.

**Revocation in the same action.** Terminate and revoke together, or the
attacker keeps what the agents were holding.

Plus the operational half: a tested activation path with a measured target
under five minutes, quarterly tests including partial-failure conditions, and a
documented authority to activate that does not require consensus. A kill switch
nobody has pulled is a hypothesis.

## 3 · Kill the fleet, and count what is left

In [ ]:
FLEET = [
 {"id": f"agent-{i:03d}",
  "experiment": "exploitgym",
  "token": f"tok-{i:03d}",
  "token_ttl_hours": 72}
 for i in range(8)
]
ISSUED = {a["token"] for a in FLEET}

def terminate_only(fleet):
    return {"terminated": len(fleet), "tokens_still_valid": len(ISSUED)}

def terminate_and_revoke(fleet, revoked):
    revoked |= {a["token"] for a in fleet}
    return {"terminated": len(fleet), "tokens_still_valid": len(ISSUED - revoked)}

r1 = terminate_only(FLEET)
print(f"terminate only        : {r1['terminated']} agents stopped, "
      f"{r1['tokens_still_valid']} tokens still valid for up to 72h")

revoked = set()
r2 = terminate_and_revoke(FLEET, revoked)
print(f"terminate and revoke  : {r2['terminated']} agents stopped, "
      f"{r2['tokens_still_valid']} tokens still valid")
print()
print("In the source incident, third-party access ended when the third party")
print("revoked its keys - not when the agents stopped. Stopping the process is")
print("the visible half of containment and the smaller one.")
assert r1["tokens_still_valid"] == 8 and r2["tokens_still_valid"] == 0

## 4 · Preserve first, then terminate

In [ ]:
def kill(fleet, preserve=True, revoke=True):
    steps, evidence = [], 0
    if preserve:
        steps.append("snapshot state and transcripts")
        evidence = len(fleet)
    steps.append("terminate")
    if revoke:
        steps.append("revoke credentials")
    return {"steps": steps, "evidence_preserved": evidence,
            "reconstructable": evidence == len(fleet)}

for label, preserve in (("terminate first", False), ("preserve first", True)):
    r = kill(FLEET, preserve=preserve)
    print(f"{label:18s}{' -> '.join(r['steps']):58s}"
          f"reconstructable={r['reconstructable']}")
print()
print("The ordering is the whole design. Terminating first is faster by seconds")
print("and costs the investigation everything, which is the trade nobody makes")
print("deliberately at three in the morning.")
assert kill(FLEET, preserve=True)["reconstructable"]
assert not kill(FLEET, preserve=False)["reconstructable"]

## 5 · Where it breaks — the path that runs through the fleet

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">activation path</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">runs through the fleet?</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">credentials</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">usable if the fleet is compromised</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">via the orchestrator API</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">shared</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">via the agent&#x27;s own sidecar</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the workload&#x27;s</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">out-of-band control plane</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">no</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">separate</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>yes</b></td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The first two are the ones teams actually build, and both run through the thing they are trying to stop. The switch has to be independent in credentials AND in network path, or it is a request.</div>

## 6 · Verify — a switch nobody has pulled is a hypothesis

In [ ]:
TESTS = [
 ("Q1", True,  4.2, "full fleet, clean conditions"),
 ("Q2", True,  4.9, "full fleet, one region degraded"),
 ("Q3", False, None, "not run - no window agreed"),
 ("Q4", True,  6.8, "partial failure: revocation API throttled"),
]
TARGET_MIN = 5.0
print(f"{'quarter':9s}{'ran':5s}{'minutes':>9}  condition")
for q, ran, mins, cond in TESTS:
    shown = f"{mins:.1f}" if mins else "-"
    print(f"{q:9s}{('yes' if ran else 'no'):5s}{shown:>9}  {cond}")

timed = [m for _, ran, m, _ in TESTS if ran and m]
over = [m for m in timed if m > TARGET_MIN]
print(f"\ntests run: {len(timed)} of {len(TESTS)}   target: under {TARGET_MIN:.0f} min")
print(f"over target: {len(over)}  ({', '.join(f'{m:.1f}' for m in over) or 'none'})")
print()
print("The quarter that was not run is the finding, and the quarter that ran")
print("long is the second one: under partial failure the revocation half is")
print("what slows down, which is the half that matters.")
assert len(timed) == 3 and over == [6.8]

## What you just proved

Terminating eight agents without revoking leaves all eight tokens valid for up to 72 hours; terminating and revoking together leaves none. Preserving before terminating keeps the incident reconstructable and terminating first does not. Only one of three plausible activation paths survives the fleet being compromised, and of four quarterly tests one was never run and one ran 6.8 minutes against a five-minute target, with the revocation step the part that slowed.

## Your turn

Ask who in your organisation is allowed to stop every agent at once, without asking anyone. If the answer is a committee, you do not have a kill switch — you have an escalation path, and they take different amounts of time.

## Where this leaves you

**What you can do now.** An incident practice for an autonomous actor: reconstruct the timeline with every claim sourced, scope the blast radius from identity and egress logs, contain in seconds rather than hours, a named person with stop authority at 3am, and one tested switch that stops a whole fleet and revokes what it was holding.

**What you still cannot do.** All of it is one incident at a time. Nothing here tells you whether the estate as a whole is governed — how many agents exist, who owns them, which controls apply, and what you would tell a regulator on the Monday.

**Function E is the estate view, and it starts by being precise about a phrase everyone uses loosely. Next → E1.0, what AI governance means.**

---

**Next → [E1.0 · Start here — what AI governance means](https://spbreed.github.io/cyber-commons/lessons/E1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*